[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elastic/elasticsearch-labs/blob/main/notebooks/integrations/gemini/audio_similarity_search_gemini_elasticsearch.ipynb)

# Audio Similarity Search with Gemini 2 Embeddings and Elasticsearch

This notebook walks through building an audio similarity search pipeline using:

- [**Gemini Embedding 2**](https://ai.google.dev/gemini-api/docs/embeddings#embed-audio) to generate vector embeddings from different modalities, such as text, images, audio, and video
- [**Elasticsearch**](https://www.elastic.co/elasticsearch) as a vector database for similarity search

Audio similarity search usually requires the audio to be converted to [spectrograms](https://en.wikipedia.org/wiki/Spectrogram) and then converted to vector embeddings using a vision encoder. (See this [audio similarity search tutorial](https://www.elastic.co/search-labs/blog/searching-by-music-leveraging-vector-search-audio-information-retrieval))

With modern omni-modal embedding models, such as Gemini Embedding 2, you can pass audio files directly to the model without prior preprocessing to spectrograms.


## 1. Prerequisites

### Install dependencies

In [4]:
%pip install -qU elasticsearch google-genai python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### API Keys

To follow this tutorial, you will need the following API keys set in your environment variables, Google Colab secrets, or `.env` file in your project root:

- `GOOGLE_API_KEY`: Go to [Google AI Studio](https://aistudio.google.com/) → **Get API key** 
- `ELASTICSEARCH_URL` and `ELASTICSEARCH_API_KEY`: see below. | Covered in the next section |

You can choose one of the two options below to connect to an Elasticsearch instance:
- **Option 1: Elastic Cloud Serverless (recommended)** Go to [cloud.elastic.co](https://cloud.elastic.co/),  create a Serverless project, **Security → API keys** 
- **Option 2: Local Docker**: Run the start-local script to spin up Elasticsearch and Kibana via Docker Compose in one command:
```bash
curl -fsSL https://elastic.co/start-local | sh
```

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")

## 2. Set up Elasticsearch

Connect to your Elasticsearch instance via the Python client.

In [6]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    hosts=[ELASTICSEARCH_URL],
    api_key=ELASTICSEARCH_API_KEY,
)

print("Connected to Elasticsearch", es.info()["version"]["number"])

Connected to Elasticsearch 9.5.0


## 3. Data Preparation

We'll be using the same small toy example dataset from  this [audio similarity search tutorial](https://www.elastic.co/search-labs/blog/searching-by-music-leveraging-vector-search-audio-information-retrieval). The dataset contains 18 `.wav` files containing the same two pieces (*Bella Ciao* and *Mozart Symphony No. 25*), each rendered in a different style (piano solo, guitar solo, and more).

In [13]:
import urllib.request
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/salgado/music-search/main/dataset/"

# Download all contents from BASE_URL
# This downloads just the .wav files found in the remote directory listing (assume known list since no directory listing available)
audio_files = [
    "a-cappella-chorus.wav",
    "bella_ciao_a-cappella-chorus.wav",
    "bella_ciao_electronic-synth-lead.wav",
    "bella_ciao_guitar-solo.wav",
    "bella_ciao_humming.wav",
    "bella_ciao_jazz-with-saxophone.wav",
    "bella_ciao_opera-singer.wav",
    "bella_ciao_piano-solo.wav",
    "bella_ciao_string-quartet.wav",
    "bella_ciao_tribal-drums-and-flute.wav",
    "mozart_symphony25_electronic-synth-lead.wav",
    "mozart_symphony25_guitar-solo.wav",
    "mozart_symphony25_jazz-with-saxophone.wav",
    "mozart_symphony25_opera-singer.wav",
    "mozart_symphony25_piano-solo.wav",
    "mozart_symphony25_prompt.wav",
    "mozart_symphony25_string-quartet.wav",
    "mozart_symphony25_tribal-drums-and-flute.wav",
]

for filename in audio_files:
    file_url = BASE_URL + filename
    dest_path = DATA_DIR / filename
    urllib.request.urlretrieve(file_url, dest_path)
    print(f"Downloaded: {filename}")

print(f"All {len(audio_files)} files downloaded to {DATA_DIR}/")

Downloaded: a-cappella-chorus.wav
Downloaded: bella_ciao_a-cappella-chorus.wav
Downloaded: bella_ciao_electronic-synth-lead.wav
Downloaded: bella_ciao_guitar-solo.wav
Downloaded: bella_ciao_humming.wav
Downloaded: bella_ciao_jazz-with-saxophone.wav
Downloaded: bella_ciao_opera-singer.wav
Downloaded: bella_ciao_piano-solo.wav
Downloaded: bella_ciao_string-quartet.wav
Downloaded: bella_ciao_tribal-drums-and-flute.wav
Downloaded: mozart_symphony25_electronic-synth-lead.wav
Downloaded: mozart_symphony25_guitar-solo.wav
Downloaded: mozart_symphony25_jazz-with-saxophone.wav
Downloaded: mozart_symphony25_opera-singer.wav
Downloaded: mozart_symphony25_piano-solo.wav
Downloaded: mozart_symphony25_prompt.wav
Downloaded: mozart_symphony25_string-quartet.wav
Downloaded: mozart_symphony25_tribal-drums-and-flute.wav
All 18 files downloaded to data/


## 4. Define the Index Mapping

We create an Elasticsearch index with two fields:

- `filename` (`keyword`): Exact identifier for the audio file
- `audio_embedding` (`dense_vector`): The vector used for kNN similarity search

Setting `similarity: cosine` tells Elasticsearch to rank results by cosine similarity. 

*Note:* We omit the `dims` parameter here. Elasticsearch will automatically infer the vector dimension from the first document indexed, so you don't need to know it upfront.

In [ ]:
INDEX_NAME = "audio-search"

# Drop the index only if it already exists so this cell is safely re-runnable
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "filename": {"type": "keyword"},
            "audio_embedding": {
                "type": "dense_vector",
                "similarity": "cosine",
            },
        }
    },
)

print(f"Index '{INDEX_NAME}' created.")

Index 'audio-search' created.


## 5. Set up `genai` Client for Gemini 2 Embeddings

Gemini Embedding 2 is a **multimodal** model: it encodes audio and text into the **same vector space**. This shared space is what makes cross-modal queries possible. For example, a text description of a mood can retrieve acoustically matching audio, and vice versa.

In this tutorial, we will focus on two modalities: text and audio. 

We define two small helper functions, one per input modality.

In [ ]:
from google import genai
from google.genai import types

gemini = genai.Client(api_key=GOOGLE_API_KEY)

def embed_audio(filepath: str) -> list[float]:
    with open(filepath, "rb") as f:
        audio_bytes = f.read()

    result = gemini.models.embed_content(
        model="gemini-embedding-2",
        contents=[
            types.Part.from_bytes(
                data=audio_bytes,
                mime_type="audio/wav",
            )
        ],
    )
    return result.embeddings[0].values


def embed_text(text: str) -> list[float]:
    result = gemini.models.embed_content(
        model="gemini-embedding-2",
        contents=text,
    )
    return result.embeddings[0].values

Let's the functions to embed the contents.

In [17]:
print("Text embedding:")
text_embedding = embed_text("Hello, world!")
print(text_embedding[:10])

print("Audio embedding:")
audio_embedding = embed_audio("data/a-cappella-chorus.wav")
print(audio_embedding[:10])

Text embedding:
[-0.022495363, 0.004016586, 0.02667887, -0.008154897, -0.018237598, 0.013646618, -0.006213285, 0.016268833, -0.005994342, -0.048850574]
Audio embedding:
[-0.035680573, 0.014103874, 0.013038295, -0.021932071, 0.013883692, 0.0040342035, -0.013924922, 0.016876323, 0.0029246118, -0.055127673]


## 6. Ingest the data

We loop over each audio file, generate an embedding, and stream the documents into Elasticsearch using the bulk helper. The generator pattern keeps memory usage flat regardless of how many files you have.

In [19]:
from elasticsearch.helpers import bulk

def generate_docs(files):
    for file in files:
        print(f"Embedding {file}...")
        embedding = embed_audio(f"data/{file}")
   
        yield {
            "_index": INDEX_NAME,
            "_id": file,
            "filename": file,
            "audio_embedding": embedding,
        }


count, _ = bulk(es, generate_docs(audio_files))
print(f"\nIndexed {count} documents.")

Embedding a-cappella-chorus.wav...
Embedding bella_ciao_a-cappella-chorus.wav...
Embedding bella_ciao_electronic-synth-lead.wav...
Embedding bella_ciao_guitar-solo.wav...
Embedding bella_ciao_humming.wav...
Embedding bella_ciao_jazz-with-saxophone.wav...
Embedding bella_ciao_opera-singer.wav...
Embedding bella_ciao_piano-solo.wav...
Embedding bella_ciao_string-quartet.wav...
Embedding bella_ciao_tribal-drums-and-flute.wav...
Embedding mozart_symphony25_electronic-synth-lead.wav...
Embedding mozart_symphony25_guitar-solo.wav...
Embedding mozart_symphony25_jazz-with-saxophone.wav...
Embedding mozart_symphony25_opera-singer.wav...
Embedding mozart_symphony25_piano-solo.wav...
Embedding mozart_symphony25_prompt.wav...
Embedding mozart_symphony25_string-quartet.wav...
Embedding mozart_symphony25_tribal-drums-and-flute.wav...

Indexed 18 documents.


## 7. Sample Search Queries

Let's test two example queries: one for text queries and one for audio queries.

Both queries use the Elasticsearch [kNN retriever](https://www.elastic.co/docs/solutions/search/vector/knn). The key parameters:

- `k`: number of results to return
- `num_candidates`: how many candidates each shard considers before returning the top `k` (higher = more accurate, slower)

### Text Search Query: (Text-to-audio)

Embed a natural language description and retrieve the most similar audio tracks. 

Below you can see that for the search query of "Jazz music", the top 2 search results are the Jazz-style rendered versions of the two different tracks.

In [25]:
query_text = "Jazz music"

response = es.search(
    index=INDEX_NAME,
    retriever={
        "knn": {
            "field": "audio_embedding",
            "query_vector": embed_text(query_text),
            "k": 3,
            "num_candidates": 5,
        }
    },
)

print(f"Top matches for '{query_text}':")
for hit in response["hits"]["hits"]:
    print(f"  [{hit['_score']:.4f}] {hit['_source']['filename']}")

Top matches for 'Jazz music':
  [0.8386] bella_ciao_jazz-with-saxophone.wav
  [0.8248] mozart_symphony25_jazz-with-saxophone.wav
  [0.8114] mozart_symphony25_guitar-solo.wav


### Audio Search Query (Audio-to-audio)

Embed a reference audio file and find the most acoustically similar tracks in the index.

Note, we're using a file that has been indexed here. Usually, search queries are not in the index. You can replace `QUERY_AUDIO_PATH` with any `.wav` file on your machine.

In [27]:
QUERY_AUDIO_PATH = str(DATA_DIR / "a-cappella-chorus.wav")

response = es.search(
    index=INDEX_NAME,
    retriever={
        "knn": {
            "field": "audio_embedding",
            "query_vector": embed_audio(QUERY_AUDIO_PATH),
            "k": 3,
            "num_candidates": 18,
        }
    },
)

print(f"Top matches for '{QUERY_AUDIO_PATH}':")
for hit in response["hits"]["hits"]:
    print(f"  [{hit['_score']:.4f}] {hit['_source']['filename']}")

Top matches for 'data/a-cappella-chorus.wav':
  [1.0000] a-cappella-chorus.wav
  [0.8834] bella_ciao_a-cappella-chorus.wav
  [0.8494] bella_ciao_humming.wav


## Summary

This tutorial gave you a quick overview of how you can do a simple multimodal search over audio data with text-to-audio and audio-to-audio queries.

### References

- [Bring your own dense vectors to Elasticsearch](https://www.elastic.co/docs/solutions/search/vector/bring-own-vectors)
- [kNN search in Elasticsearch](https://www.elastic.co/docs/solutions/search/vector/knn)
- [Dense vector field type reference](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/dense-vector)
- [Gemini Embedding API](https://ai.google.dev/gemini-api/docs/embeddings)